# 6b · Connect an MCP client + security

**Core · Live-code · ~25 min**

Now we play the other role: the **client**. We'll launch the server, discover its tools *over
the protocol*, and call them — exactly as VS Code or an AI assistant would. This is the moment
MCP stops being abstract and becomes real.

**Note:** these cells use top-level `await`, which works in Jupyter. Solution:
[`solution/06b_connect_client.ipynb`](solution/06b_connect_client.ipynb).

## Step 1 — Point the client at the server

With **stdio** transport, the client *launches the server as a subprocess* and talks to it
through the pipe. So we describe how to start it: run `python orbital_mcp_server.py`. We use an
absolute path so it works no matter where the notebook is run from.

In [ ]:
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# This tells the client how to start our server.
server = StdioServerParameters(command="python", args=[os.path.abspath("orbital_mcp_server.py")])

## Step 2 — Connect and discover the tools

The `async with` blocks open the connection and a **session**, then `initialize()` performs
the MCP handshake. After that, `list_tools()` asks the server what it can do — the client
*discovers* the tools at runtime rather than having them hard-coded. That dynamic discovery is
a big part of why MCP is powerful.

(The nested `async with` / `await` style is just how we talk to an async protocol — you can
follow the pattern without being an async expert.)

In [ ]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        result = await session.list_tools()
        print("Tools the server offers:")
        for t in result.tools:
            print("-", t.name)

## Step 3 — Call a tool over the protocol

Now actually *use* a tool. `session.call_tool(name, arguments)` sends the request to the
server, which runs the function and returns the result. We read the text out of the response.

This is the same `get_sensor` you inspected in 6a — but now it's being invoked across the MCP
boundary, exactly as an external app would do it.

In [ ]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        # TODO: call the 'get_sensor' tool for the 'o2_pct' signal and print the text result.
        #   res = await session.call_tool("get_sensor", {"signal": "o2_pct"})
        #   print(res.content[0].text)

## Step 4 — Security check: the guarded action

Let's confirm the safety design holds *over MCP too*. Call `control_valve` and verify the
server refuses to make a real change and instead asks for human confirmation. No matter who
the client is, the guard travels with the server.

In [ ]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        res = await session.call_tool("control_valve", {"valve": "vent_b", "state": "open"})
        print(res.content[0].text)

### 🌱 Stretch (optional)

- Call `get_sensor` with an invalid signal (e.g. `"banana"`) and watch the server validate the
  input and return a helpful error rather than crashing.
- Peek ahead: in Module 7, ARIA (the agent) will connect to this very server and use these
  tools as its hands to resolve the O₂ emergency.

## ✅ Recap

You connected as an MCP client, discovered the server's tools at runtime, called them over the
protocol, and confirmed the safety guard travels with the server. ARIA's abilities are now a
standard service any AI app can use — which is exactly what we'll take advantage of in the
capstone.

# 6b · Connect an MCP client + security

**Core · Live-code · ~25 min**

Now we act as an MCP **client**: launch the server, discover its tools, and call them over
the protocol.

> Uses top-level `await` (works in Jupyter). Solution: [`solution/06b_connect_client.ipynb`](solution/06b_connect_client.ipynb).

In [ ]:
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# The client launches the server as a subprocess and talks to it over stdio.
server = StdioServerParameters(command="python", args=[os.path.abspath("orbital_mcp_server.py")])

## Discover the tools

In [ ]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        result = await session.list_tools()
        for t in result.tools:
            print("-", t.name)

## Call a tool

Ask the server for the latest oxygen reading.

In [ ]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        # TODO: call the 'get_sensor' tool with {"signal": "o2_pct"} and print the text result
        # res = await session.call_tool("get_sensor", {"signal": "o2_pct"})
        # print(res.content[0].text)

## Security check: the guarded action

Call `control_valve` and confirm the server refuses to make a real change without a human.

In [ ]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        res = await session.call_tool("control_valve", {"valve": "vent_b", "state": "open"})
        print(res.content[0].text)

### 🌱 Stretch
- Try calling `get_sensor` with an invalid signal — see the server validate the input.
- In Module 7, ARIA (the agent) will use this MCP server as its hands to resolve an incident.